# Optimizing the Piezo Position (Focus-Based)

``` Pseudo
1. Get scanners current position
2. Set resolution (we know range will be 1 um in the xy plane)
3. Build the sequence, centered around the current position
4. For each position in sequence, go to that position and take an image
5. Calculate the focus sharpness for each image, and fit a gaussian distribution to the sequence
6. Plot
```

### Scanner Architecture: GUI to Hardware

- **For positioning**: ScannerGui → ScanningProbeLogic → NiScanningProbeInterfuse → **ni_ao** → Hardware
- **For scanning**: ScannerGui → ScanningProbeLogic → NiScanningProbeInterfuse → **ni_finite_sampling_io** → Hardware
- Interfuse selects appropriate hardware
Piezo stages: x, y, z axes
    - ao0 → x-axis
    - ao1 → y-axis  
    - ao2 → z-axis

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
import cv2
import torch
import torch.nn.functional as F

## SPAD Configuration

In [ ]:
spad = camera_SPC3
spad._NFrames = 1 
spad._apply_camera_settings()

## Building the Sequence

### Optimization Params
- resolution_x = 10 # pixels
- resolution_y = 10 # pixels
- scan_range_x = 1e-6 # units are in m; 1 micron for both x and y ensures pixel is centered on nv
- scan_range_y = 1e-6

In [ ]:
resolution_x = 10 # pixels
resolution_y = 10 # pixels
scan_range_x = 1.0e-6 # meters
scan_range_y = 1.0e-6 # meters
arr_dim = 5 # what are the number of nv's in each row and column of the array

piezo = scanning_probe_logic
x_l_bound, x_u_bound = piezo.scanner_constraints.axes['x'].position.bounds
y_l_bound, y_u_bound = piezo.scanner_constraints.axes['y'].position.bounds

# Build sequence
curr_pos = piezo.scanner_position
start_x = curr_pos['x'] - scan_range_x / 2
start_y = curr_pos['y'] - scan_range_y / 2
sequence_x = np.linspace(start_x, start_x + scan_range_x, int(resolution_x))
sequence_y = np.linspace(start_y, start_y + scan_range_y, int(resolution_y))

# Check params
w, h = spad.get_size()
array_size_microns = 2 * arr_dim - 1 
pxl_per_micron_x = w / array_size_microns  # 32 pixels / 9 microns = 3.56 pxl/micron
pxl_per_micron_y = h / array_size_microns

if sequence_x[0] < x_l_bound or sequence_x[-1] > x_u_bound:
    raise ValueError(f"X scan range [{sequence_x[0]:.2e}, {sequence_x[-1]:.2e}] m exceeds scanner bounds [{x_l_bound:.2e}, {x_u_bound:.2e}] m")

if sequence_y[0] < y_l_bound or sequence_y[-1] > y_u_bound:
    raise ValueError(f"Y scan range [{sequence_y[0]:.2e}, {sequence_y[-1]:.2e}] m exceeds scanner bounds [{y_l_bound:.2e}, {y_u_bound:.2e}] m")

print(curr_pos)

## Sampling Images in the XY-Plane

### Position Dictionary
```
{'x': 0.0001729267, 'y': 2.8033e-05, 'z': 0.0001018735}
```

In [ ]:
img_samples = np.zeros((resolution_x * resolution_y, h, w), dtype=np.float32)
pos_dict = {k: float(v) for k, v in curr_pos.items()}

idx = 0
for y_sample in sequence_y:
    for x_sample in sequence_x:
        pos_dict['x'] = float(x_sample)
        pos_dict['y'] = float(y_sample)
        
        position = piezo.set_target_position(pos_dict, move_blocking=True)
        
        frame = np.array(spad.start_single_acquisition()[0, 0, :, :])
        img_samples[idx] = frame.astype(np.float32)
        idx += 1
        print(f"Captured frame {idx}/{resolution_x * resolution_y}")

print(f"Final shape: {img_samples.shape}")

## Focus Metric Functions

In [ ]:
img_samples = np.nan_to_num(img_samples, 0)

def sum_array_batched(imgs, pxl_per_micron_x, pxl_per_micron_y):
    """Apply kernel convolution to extract NV features at grid positions."""
    imgs = torch.from_numpy(imgs).float().unsqueeze(1)

    nv_size_x = int(pxl_per_micron_x)  # pixels per NV
    nv_size_y = int(pxl_per_micron_y)

    # Kernel stride
    spacing_x = int(2 * pxl_per_micron_x)  
    spacing_y = int(2 * pxl_per_micron_y)
    stride = (spacing_y, spacing_x)

    # Kernel = 1 micron NV box
    kernel = torch.ones((1, 1, nv_size_y, nv_size_x), dtype=imgs.dtype)

    out = F.conv2d(imgs, kernel, stride=stride, padding=0)
    brightness = out.squeeze(1).cpu().numpy()
    return brightness

def laplacian_variance_grid(imgs, roi_fraction=0.8):
    """
    Uses region of interest (ROI) to focus on center and larger kernel for better grid edge detection.
    
    Args:
        imgs: Array of images (N, H, W)
        roi_fraction: Fraction of image to use (0.8 = use center 80%)
    """
    sharpness = np.zeros(imgs.shape[0])
    h, w = imgs.shape[1], imgs.shape[2]
    
    roi_h = int(h * roi_fraction)
    roi_w = int(w * roi_fraction)
    y_start = (h - roi_h) // 2
    x_start = (w - roi_w) // 2
    
    for i in range(imgs.shape[0]):
        roi = imgs[i, y_start:y_start+roi_h, x_start:x_start+roi_w]
        
        img_uint8 = np.clip(roi, 0, 255).astype(np.uint8)
        
        laplacian = cv2.Laplacian(img_uint8, cv2.CV_64F, ksize=5)
        
        sharpness[i] = laplacian.var()
    
    return sharpness

def kernel_based_focus(imgs, pxl_per_micron_x, pxl_per_micron_y):
    """
    Focus metric based on NV grid kernel convolution.
    More accurate for grid patterns - focuses on actual NV positions.
    
    Args:
        imgs: Array of images (N, H, W)
        pxl_per_micron_x, pxl_per_micron_y: Kernel size parameters
    
    Returns:
        Sharpness metric for each image
    """
    nv_features = sum_array_batched(imgs, pxl_per_micron_x, pxl_per_micron_y)
    
    sharpness = np.var(nv_features, axis=(1, 2))
    
    return sharpness

# Calculate focus metrics using BOTH methods for comparison
focus_kernel = kernel_based_focus(img_samples, pxl_per_micron_x, pxl_per_micron_y)
focus_laplacian_grid = laplacian_variance_grid(img_samples, roi_fraction=0.8)

print(f"Original: {img_samples.shape}")
print(f"Kernel focus range: [{focus_kernel.min():.2f}, {focus_kernel.max():.2f}]")
print(f"Laplacian focus range: [{focus_laplacian_grid.min():.2f}, {focus_laplacian_grid.max():.2f}]")

## Reshape Focus Metrics to 2D Grid

In [ ]:
# Reshape to 2D grid: (resolution_y, resolution_x)
focus_kernel_2d = np.reshape(focus_kernel, (resolution_y, resolution_x))
focus_laplacian_2d = np.reshape(focus_laplacian_grid, (resolution_y, resolution_x))

## Compare Focus Methods

In [ ]:
# Find peaks for each method
row_kernel, col_kernel = np.unravel_index(np.argmax(focus_kernel_2d), focus_kernel_2d.shape)
row_lap, col_lap = np.unravel_index(np.argmax(focus_laplacian_2d), focus_laplacian_2d.shape)

X_grid, Y_grid = np.meshgrid(sequence_x, sequence_y, indexing="xy")
pos_x_kernel = X_grid[row_kernel, col_kernel]
pos_y_kernel = Y_grid[row_kernel, col_kernel]
pos_x_lap = X_grid[row_lap, col_lap]
pos_y_lap = Y_grid[row_lap, col_lap]

# Calculate SNR for each method
snr_kernel = (focus_kernel.max() - focus_kernel.min()) / np.std(focus_kernel)
snr_lap = (focus_laplacian_grid.max() - focus_laplacian_grid.min()) / np.std(focus_laplacian_grid)

print("\n=== Focus Method Comparison ===")
print(f"Kernel-based: Peak at ({pos_x_kernel*1e6:.3f}, {pos_y_kernel*1e6:.3f}) µm, SNR: {snr_kernel:.2f}")
print(f"Laplacian Grid: Peak at ({pos_x_lap*1e6:.3f}, {pos_y_lap*1e6:.3f}) µm, SNR: {snr_lap:.2f}")

# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Kernel-based
im0 = axes[0].imshow(focus_kernel_2d, origin='lower', aspect='auto')
axes[0].plot(col_kernel, row_kernel, 'r+', markersize=15, markeredgewidth=2)
axes[0].set_title(f'Kernel-Based Focus\nSNR: {snr_kernel:.2f}')
axes[0].set_xlabel('X index')
axes[0].set_ylabel('Y index')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

# Laplacian Grid
im1 = axes[1].imshow(focus_laplacian_2d, origin='lower', aspect='auto')
axes[1].plot(col_lap, row_lap, 'r+', markersize=15, markeredgewidth=2)
axes[1].set_title(f'Laplacian Grid Focus\nSNR: {snr_lap:.2f}')
axes[1].set_xlabel('X index')
axes[1].set_ylabel('Y index')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

# SNR comparison
axes[2].bar(['Kernel-Based', 'Laplacian Grid'], [snr_kernel, snr_lap])
axes[2].set_ylabel('Signal-to-Noise Ratio')
axes[2].set_title('Focus Method SNR Comparison')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Select Best Focus Method and View Image at Peak

In [ ]:
# Choose the method with best SNR (or manually select)
if snr_kernel > snr_lap:
    print("Using Kernel-Based Focus (better SNR)")
    focus_metric = focus_kernel_2d
    row_max, col_max = row_kernel, col_kernel
else:
    print("Using Laplacian Grid Focus (better SNR)")
    focus_metric = focus_laplacian_2d
    row_max, col_max = row_lap, col_lap

pos_x_max = X_grid[row_max, col_max]
pos_y_max = Y_grid[row_max, col_max]

# Move to best position and capture averaged frames
pos_dict['x'] = float(pos_x_max)
pos_dict['y'] = float(pos_y_max)

position = piezo.set_target_position(pos_dict, move_blocking=True)

frames = []
for i in range(5):
    single_frame = np.array(spad.start_single_acquisition()[0, 0, :, :])
    frames.append(single_frame)
    print(f"Captured frame {i+1}/5")

frame_avg = np.mean(frames, axis=0)
focus_value = focus_metric[row_max, col_max]

plt.figure(figsize=(8, 6))
plt.imshow(np.flipud(frame_avg), origin='lower')
plt.colorbar(label='Counts')
plt.title(f'Best Focus Position: ({pos_x_max*1e6:.3f}, {pos_y_max*1e6:.3f}) µm\nFocus Metric: {focus_value:.2f}')
plt.xlabel('X (pixels)')
plt.ylabel('Y (pixels)')
plt.show()

print(f"Position indices: row={row_max}, col={col_max}")
print(f"Physical position: x={pos_x_max*1e6:.3f} µm, y={pos_y_max*1e6:.3f} µm")
print(f"Focus metric value: {focus_value:.2f}")

## Fit 2D Gaussian to Focus Data

In [ ]:
def gaussian_2d(XY, amplitude, x_center, y_center, sigma_x, sigma_y, background):
    x, y = XY
    g = background + amplitude * np.exp(
        -(((x - x_center) ** 2) / (2 * sigma_x ** 2) +
          ((y - y_center) ** 2) / (2 * sigma_y ** 2))
    )
    return g.ravel()

X, Y = np.meshgrid(sequence_x, sequence_y, indexing="xy")
z = focus_metric.astype(float)

row_max, col_max = np.unravel_index(np.argmax(z), z.shape)
x_center_guess = X[row_max, col_max]
y_center_guess = Y[row_max, col_max]
background_guess = float(np.min(z))
amplitude_guess = float(z[row_max, col_max] - background_guess)

scan_range_x_local = float(sequence_x.max() - sequence_x.min())
scan_range_y_local = float(sequence_y.max() - sequence_y.min())
sigma_x_guess = scan_range_x_local / 4.0 if scan_range_x_local > 0 else 1.0
sigma_y_guess = scan_range_y_local / 4.0 if scan_range_y_local > 0 else 1.0

p0 = (amplitude_guess, x_center_guess, y_center_guess, sigma_x_guess, sigma_y_guess, background_guess)

bounds = (
    [0.0, sequence_x.min(), sequence_y.min(), 1e-12, 1e-12, -np.inf],
    [np.inf, sequence_x.max(), sequence_y.max(), np.inf, np.inf,  np.inf],
)

popt, _ = curve_fit(
    gaussian_2d,
    (X, Y),
    z.ravel(),
    p0=p0,
    bounds=bounds
)

amplitude_fit, x_center_fit, y_center_fit, sigma_x_fit, sigma_y_fit, background_fit = popt

pred = (background_fit + amplitude_fit * np.exp(
    -(((X - x_center_fit) ** 2) / (2 * sigma_x_fit ** 2) +
      ((Y - y_center_fit) ** 2) / (2 * sigma_y_fit ** 2))
))

resid = z - pred

sse = float(np.sum(resid**2))
sst = float(np.sum((z - np.mean(z))**2))
r2 = 1.0 - (sse / sst) if sst > 0 else float("nan")
rmse = float(np.sqrt(np.mean(resid**2)))

print("Peak location:", x_center_fit, y_center_fit)
print("amplitude, background:", amplitude_fit, background_fit)
print("sigma_x, sigma_y:", sigma_x_fit, sigma_y_fit)
print(f"R-Squared: {r2:.3f}")
print(f"RMSE: {rmse:.3f}")

# Move to fitted optimal position and capture
pos_dict["x"], pos_dict["y"] = float(x_center_fit), float(y_center_fit)
position = piezo.set_target_position(pos_dict, move_blocking=True)

frames = []
for i in range(5):
    single_frame = np.array(spad.start_single_acquisition()[0, 0, :, :])
    frames.append(single_frame)
    print(f"Captured frame {i+1}/5")

frame_avg = np.mean(frames, axis=0)

plt.figure(figsize=(8, 6))
plt.imshow(np.flipud(frame_avg), origin='lower')
plt.colorbar(label='Counts')
plt.title(f'Gaussian Fit Position: ({x_center_fit*1e6:.3f}, {y_center_fit*1e6:.3f}) µm\n(avg of 5 frames)')
plt.xlabel('X (pixels)')
plt.ylabel('Y (pixels)')
plt.show()

## Plot Gaussian Fit Quality

In [ ]:
plt.figure(figsize=(12, 4))

vmin = min(z.min(), pred.min())
vmax = max(z.max(), pred.max())

plt.subplot(1, 3, 1)
plt.title("Measured Focus")
plt.imshow(z, origin="lower", vmin=vmin, vmax=vmax)
plt.colorbar(fraction=0.046)

plt.subplot(1, 3, 2)
plt.title("Predicted (Gaussian fit)")
plt.imshow(pred, origin="lower", vmin=vmin, vmax=vmax)
plt.colorbar(fraction=0.046)

plt.subplot(1, 3, 3)
plt.title("Residual (meas - pred)")
m = np.max(np.abs(resid))
plt.imshow(resid, origin="lower", vmin=-m, vmax=m, cmap='RdBu')
plt.colorbar(fraction=0.046)

plt.tight_layout()
plt.show()